# 3DGS Pipeline: Video → Gaussian Splat (.ply)

Video (mp4) → frame extraction + COLMAP (SfM) → splatfacto training → splat.ply → download

Run cells in order. Do not skip the verification cells (⚠️) — they fail fast with a clear
message instead of letting you discover problems 30 minutes in.

In [2]:
%%bash
set -e
export DEBIAN_FRONTEND=noninteractive

apt-get update -qq
apt-get install -y -qq ffmpeg colmap ninja-build build-essential git libgl1 libglib2.0-0 zip

# Venv inherits the container's native PyTorch/CUDA
python -m venv --system-site-packages /workspace/venv_3dgs
source /workspace/venv_3dgs/bin/activate

pip install --upgrade pip setuptools wheel

# Fallback: only installs torch if the pod image does NOT provide one
if ! python -c "import torch" 2>/dev/null; then
  echo ">>> No system PyTorch found — installing cu121 build into venv"
  pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
fi

pip install --no-cache-dir nerfstudio
echo ">>> Installation complete"

debconf: delaying package configuration, since apt-utils is not installed


(Reading database ... 24135 files and directories currently installed.)
Preparing to unpack .../0-libglib2.0-bin_2.72.4-0ubuntu2.9_amd64.deb ...
Unpacking libglib2.0-bin (2.72.4-0ubuntu2.9) over (2.72.4-0ubuntu2.3) ...
Preparing to unpack .../1-libglib2.0-0_2.72.4-0ubuntu2.9_amd64.deb ...
Unpacking libglib2.0-0:amd64 (2.72.4-0ubuntu2.9) over (2.72.4-0ubuntu2.3) ...
Selecting previously unselected package shared-mime-info.
Preparing to unpack .../2-shared-mime-info_2.1-2_amd64.deb ...
Unpacking shared-mime-info (2.1-2) ...
Preparing to unpack .../3-libatomic1_12.3.0-1ubuntu1~22.04.3_amd64.deb ...
Unpacking libatomic1:amd64 (12.3.0-1ubuntu1~22.04.3) over (12.3.0-1ubuntu1~22.04) ...
Preparing to unpack .../4-libubsan1_12.3.0-1ubuntu1~22.04.3_amd64.deb ...
Unpacking libubsan1:amd64 (12.3.0-1ubuntu1~22.04.3) over (12.3.0-1ubuntu1~22.04) ...
Preparing to unpack .../5-libquadmath0_12.3.0-1ubuntu1~22.04.3_amd64.deb ...
Unpacking libquadmath0:amd64 (12.3.0-1ubuntu1~22.04.3) over (12.3.0-1ubuntu

In [3]:
%%bash
source /workspace/venv_3dgs/bin/activate

python - <<'PY'
import torch
print(f"PyTorch {torch.__version__} | CUDA build: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} | VRAM: {p.total_memory/2**30:.1f} GB")
import nerfstudio, gsplat
print(f"nerfstudio OK | gsplat {gsplat.__version__}")
PY

ffmpeg -version | head -n 1
colmap -h | head -n 1
ns-train --help > /dev/null 2>&1 && echo "ns-train CLI OK"

PyTorch 2.4.1+cu124 | CUDA build: 12.4
CUDA available: True
GPU: NVIDIA RTX A6000 | VRAM: 47.4 GB
nerfstudio OK | gsplat 1.4.0
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
COLMAP 3.7 -- Structure-from-Motion and Multi-View Stereo
ns-train CLI OK


In [4]:
%%bash
mkdir -p /workspace/projeto_3dgs/input /workspace/projeto_3dgs/output
ls -la /workspace/projeto_3dgs/

total 0
drwxr-xr-x 4 root root  33 Sep 15 14:40 .
drwxr-xr-x 5 root root 111 Sep 15 14:40 ..
drwxr-xr-x 2 root root   6 Sep 15 14:40 input
drwxr-xr-x 2 root root   6 Sep 15 14:40 output


## Upload your video — 3 options

**A) Drag & drop (recommended for large files):** in the JupyterLab file browser, navigate to
`/workspace/projeto_3dgs/input/`, drag the video in, and rename it to exactly `video.mp4`.

**B) Widget (next cell):** fine for files up to ~500 MB.

**C) Direct URL:**
    curl -L "YOUR_URL_HERE" -o /workspace/projeto_3dgs/input/video.mp4

In [5]:
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

DEST = Path("/workspace/projeto_3dgs/input/video.mp4")

uploader = widgets.FileUpload(accept="video/*", multiple=False)
display(uploader)

def _save(change):
    v = uploader.value
    if not v:
        return
    if isinstance(v, dict):            # ipywidgets 7
        name, meta = next(iter(v.items()))
    else:                              # ipywidgets 8 (tuple of dicts)
        meta, name = v[0], v[0]["name"]
    content = meta["content"]
    data = content.tobytes() if hasattr(content, "tobytes") else content
    DEST.write_bytes(data)
    print(f"✅ '{name}' saved to {DEST} ({len(data)/1e6:.1f} MB)")

uploader.observe(_save, names="value")

FileUpload(value=(), accept='video/*', description='Upload')

In [6]:
%%bash
VIDEO=/workspace/projeto_3dgs/input/video.mp4
if [ ! -s "$VIDEO" ]; then
  echo "❌ video.mp4 not found. Upload it first (Cell 5)."
  exit 1
fi
ls -lh "$VIDEO"
ffprobe -v error \
  -show_entries format=duration:stream=codec_name,width,height,r_frame_rate \
  -of default=noprint_wrappers=1 "$VIDEO"

-rw-r--r-- 1 root root 71M Sep 15 14:42 /workspace/projeto_3dgs/input/video.mp4
codec_name=h264
width=1920
height=1080
r_frame_rate=30/1
codec_name=aac
r_frame_rate=0/0
duration=34.648500


In [8]:
%%bash
set -e
source /workspace/venv_3dgs/bin/activate
cd /workspace/projeto_3dgs

# Clean the partial state from the failed run
rm -rf output/dataset_formatado

ns-process-data video \
  --data input/video.mp4 \
  --output-dir output/dataset_formatado \
  --num-frames-target 300 \
  --no-gpu

echo "------------------------------------------"
echo "Frames in dataset: $(ls output/dataset_formatado/images | wc -l)"
if [ -f output/dataset_formatado/transforms.json ]; then
  echo "✅ transforms.json generated — COLMAP succeeded"
else
  echo "❌ transforms.json missing — COLMAP failed (run the diagnostic cell below)"
  exit 1
fi

Number of frames in video: 1036ages...
Extracting 346 frames in evenly spaced intervals
[14:46:09] 🎉 Done converting video to images.                                                 ]8;id=7932213;file:///workspace/venv_3dgs/lib/python3.11/site-packages/nerfstudio/process_data/process_data_utils.py\process_data_utils.py]8;;\:]8;id=7932214;file:///workspace/venv_3dgs/lib/python3.11/site-packages/nerfstudio/process_data/process_data_utils.py#227\227]8;;\
(●     ) Converting video to images...
🌕  Running COLMAP feature extractor...tor...
[14:47:43] 🎉 Done extracting COLMAP features.                                                       ]8;id=7932221;file:///workspace/venv_3dgs/lib/python3.11/site-packages/nerfstudio/process_data/colmap_utils.py\colmap_utils.py]8;;\:]8;id=7932222;file:///workspace/venv_3dgs/lib/python3.11/site-packages/nerfstudio/process_data/colmap_utils.py#137\137]8;;\
🏃  Running COLMAP feature matcher...0m
[14:58:52] 🎉 Done matching COLMAP features.      

## Training (splatfacto)

- Default: 30,000 iterations with adaptive density control (split/clone/prune).
- `cull-alpha-thresh 0.005` prunes near-invisible Gaussians → cleaner, lighter .ply.
- Approximate time for 300 frames:

  | GPU | Expected time |
  |---|---|
  | RTX 4090 / A100 | ~15–30 min |
  | RTX 3090 / A5000 | ~30–50 min |

- If the browser disconnects, training keeps running on the pod; checkpoints are saved under
  `output/treinamento/splatfacto/<timestamp>/`.
- Optional live monitoring: expose TCP 6006 and run
  `tensorboard --logdir output/treinamento --host 0.0.0.0 --port 6006`
  (or replace `--vis tensorboard` with `--vis viewer` and expose TCP 7007).

In [9]:
%%bash
set -e
source /workspace/venv_3dgs/bin/activate
cd /workspace/projeto_3dgs

ns-train splatfacto \
  --data output/dataset_formatado \
  --output-dir output/treinamento \
  --vis tensorboard \
  --pipeline.model.cull-alpha-thresh 0.005

1210 (4.03%)        26.116 ms            12 m, 31 s           28.32 M                                    
Step (% Done)       Train Iter (time)    ETA (time)           Train Rays / Sec     Test Rays / Sec       
-------------------------------------------------------------------------------------------------------- 
1130 (3.77%)        15.405 ms            7 m, 24 s            33.86 M                                    
1140 (3.80%)        15.776 ms            7 m, 35 s            33.07 M                                    
1150 (3.83%)        15.588 ms            7 m, 29 s            33.50 M                                    
1160 (3.87%)        15.517 ms            7 m, 27 s            33.64 M                                    
1170 (3.90%)        15.662 ms            7 m, 31 s            33.37 M                                    
1180 (3.93%)        14.895 ms            7 m, 9 s             35.03 M                                    
1190 (3.97%)        14.837 ms            7 m, 

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Step (% Done)       Train Iter (time)    ETA (time)           Train Rays / Sec     Test Rays / Sec       
-------------------------------------------------------------------------------------------------------- 
28910 (96.37%)      25.944 ms            28 s, 279.422 ms     29.23 M                                    
28920 (96.40%)      26.041 ms            28 s, 124.609 ms     28.92 M                                    
28930 (96.43%)      16.836 ms            18 s, 14.724 ms      31.28 M                                    
28940 (96.47%)      16.842 ms            17 s, 852.541 ms     31.23 M                                    
28950 (96.50%)      16.902 ms            17 s, 747.194 ms     31.00 M                                    
28960 (96.53%)      16.997 ms            17 s, 676.933 ms     30.87 M                                    
28970 (96.57%)      16.944 ms            17 s, 452.704 ms     30.92 M                                    
28980 (96.60%)      16.749 ms            17 s,

In [10]:
%%bash
set -e
source /workspace/venv_3dgs/bin/activate
cd /workspace/projeto_3dgs

CONFIG_PATH=$(find output/treinamento/ -name "config.yml" | sort -r | head -n 1)
if [ -z "$CONFIG_PATH" ]; then
  echo "❌ config.yml not found — did training finish?"
  exit 1
fi
echo "Using config: $CONFIG_PATH"

ns-export gaussian-splat \
  --load-config "$CONFIG_PATH" \
  --output-dir output/exportacao_final

ls -lh output/exportacao_final/

Using config: output/treinamento/dataset_formatado/splatfacto/2026-09-15_163047/config.yml


/workspace/venv_3dgs/lib/python3.11/site-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/workspace/venv_3dgs/lib/python3.11/site-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/workspace/venv_3dgs/lib/python3.11/site-packages/torchmetrics/functional/image/lpips.py:332: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default 

Unable to load the following plugins:

	libfilter_ao.so: libfilter_ao.so does not seem to be a Qt Plugin.

Cannot load library /workspace/venv_3dgs/lib/python3.11/site-packages/pymeshlab/lib/plugins/libfilter_ao.so: (libOpenGL.so.0: cannot open shared object file: No such file or directory)
	libfilter_meshing.so: libfilter_meshing.so does not seem to be a Qt Plugin.

Cannot load library /workspace/venv_3dgs/lib/python3.11/site-packages/pymeshlab/lib/plugins/libfilter_meshing.so: (libOpenGL.so.0: cannot open shared object file: No such file or directory)
	libfilter_plymc.so: libfilter_plymc.so does not seem to be a Qt Plugin.

Cannot load library /workspace/venv_3dgs/lib/python3.11/site-packages/pymeshlab/lib/plugins/libfilter_plymc.so: (libOpenGL.so.0: cannot open shared object file: No such file or directory)
	libfilter_sample_gpu.so: libfilter_sample_gpu.so does not seem to be a Qt Plugin.

Cannot load library /workspace/venv_3dgs/lib/python3.11/site-packages/pymeshlab/lib/plugins/li

/workspace/venv_3dgs/lib/python3.11/site-packages/nerfstudio/utils/eval_utils.py:62: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded_state = torch.load(load_path, map_l

✅ Done loading checkpoint from 
output/treinamento/dataset_formatado/splatfacto/2026-09-15_163047/nerfstudio_models/step-000029999.ckpt
0 Gaussians have NaN/Inf and 30013 have low opacity, only export 393598/423611
total 94M
-rw-r--r-- 1 root root 94M Sep 15 16:52 splat.ply


In [11]:
import os

ply_path = "/workspace/projeto_3dgs/output/exportacao_final/splat.ply"

if os.path.exists(ply_path):
    size_mb = os.path.getsize(ply_path) / 1e6
    header = []
    with open(ply_path, "rb") as f:
        for line in f:
            decoded = line.decode("utf-8").strip()
            header.append(decoded)
            if decoded == "end_header":
                break
    print("--- .PLY HEADER ---")
    print("\n".join(header))
    for line in header:
        if line.startswith("element"):
            print(f"\n✅ {int(line.split()[-1]):,} Gaussians | {size_mb:,.1f} MB")
else:
    print("❌ splat.ply not found.")

--- .PLY HEADER ---
ply
format binary_little_endian 1.0
comment Generated by Nerstudio 1.1.5
comment Vertical Axis: z
element vertex 393598
property float x
property float y
property float z
property float nx
property float ny
property float nz
property float f_dc_0
property float f_dc_1
property float f_dc_2
property float f_rest_0
property float f_rest_1
property float f_rest_2
property float f_rest_3
property float f_rest_4
property float f_rest_5
property float f_rest_6
property float f_rest_7
property float f_rest_8
property float f_rest_9
property float f_rest_10
property float f_rest_11
property float f_rest_12
property float f_rest_13
property float f_rest_14
property float f_rest_15
property float f_rest_16
property float f_rest_17
property float f_rest_18
property float f_rest_19
property float f_rest_20
property float f_rest_21
property float f_rest_22
property float f_rest_23
property float f_rest_24
property float f_rest_25
property float f_rest_26
property float f_rest_27

In [12]:
%%bash
set -e
cd /workspace/projeto_3dgs/output
zip -9 -j splat_final.zip exportacao_final/splat.ply
ls -lh splat_final.zip
echo ""
echo ">>> File ready: /workspace/projeto_3dgs/output/splat_final.zip"

  adding: splat.ply (deflated 12%)
-rw-r--r-- 1 root root 82M Sep 15 16:53 splat_final.zip

>>> File ready: /workspace/projeto_3dgs/output/splat_final.zip


## Downloading the result

**Option A — JupyterLab (easiest):** file browser → `/workspace/projeto_3dgs/output/` →
right-click `splat_final.zip` → **Download**.

**Option B — runpodctl (fast for large files):**
    runpodctl send /workspace/projeto_3dgs/output/splat_final.zip
then on your local machine: `runpodctl receive <code>`.

**Option C — HTTP server:** expose TCP 8000 on the pod, then
    python -m http.server 8000 --directory /workspace/projeto_3dgs/output

## Cleanup (billing!)

- **Stop pod** = stops compute charges; volume persists (small storage fee). Files survive.
- **Terminate pod** = deletes everything unless you're on a network volume.

## Next step

Drag `splat.ply` into https://superspl.at to inspect, clean, and re-export.